# 节点 5：Tool Calling 与洗晒 Agent

这一节点让助手不再依赖页面提前拼好全部资料，而是能按请求选择天气或物品工具；任何记录写入都必须再点一次确认。

## 1. 本节点目标

实现三个边界清楚的工具、有限轮次 Agent 和规则降级。同时把周期改为可选，并补充排序、分类筛选、管理搜索和状态卡片快捷编辑。

## 2. 完成结果与验收

- `get_weather` 和 `get_item_records` 只读。
- `complete_laundry_task` 默认只产生待确认动作。
- 工具参数经过 Pydantic 校验，未知工具不会执行。
- Agent 最多运行 5 轮；无密钥或失败时回到规则建议。
- 周期可留空，未设置时仍显示距上次天数但不提醒。
- 支持排序、分类筛选和状态卡片快捷编辑；添加物品使用独立入口。
- 计划类请求必须同时取得天气和物品事实。
- 雨天或高湿时不会建议户外晾晒。
- 全量 57 项测试通过。

## 3. 本节点文件结构

```text
src/smart_laundry/tools.py       工具 schema、参数校验和执行边界
src/smart_laundry/agent.py       有限轮次工具调用与规则降级
src/smart_laundry/item_views.py  排序、筛选和搜索纯函数
src/smart_laundry/database.py    可选周期的兼容迁移
app.py                           Agent、确认按钮与新交互
tests/test_tools.py              工具与确认机制测试
tests/test_agent.py              Agent 循环和降级测试
notebooks/06_tool_calling_agent.ipynb
```

## 4. 关键代码解释

`LaundryToolRegistry.execute()` 是唯一工具入口。它先根据工具名选择参数模型，再调用天气服务或 repository。写工具收到 `allow_write=False` 时只返回 `pending_action`，不会修改数据库。

`run_laundry_agent()` 把工具定义交给 Responses API，读取返回的 `function_call`，执行本地工具，再用相同 `call_id` 加入 `function_call_output`。没有函数调用时才读取最终文本。循环上限防止模型持续调用。

In [ ]:
def write_decision(allow_write):
    return '写入数据库' if allow_write else '只生成待确认动作'

print(write_decision(False))
print(write_decision(True))

## 5. 数据流

```mermaid
flowchart LR
 A[用户自然语言] --> B[模型选择工具]
 B --> C[参数校验]
 C --> D[天气/物品只读工具]
 C --> E[写工具]
 D --> F[function_call_output]
 E --> G[待确认动作]
 G -->|用户点击确认| H[事务写入 SQLite]
 F --> B
 B --> I[最终计划]
```

## 6. 关键概念

- **Tool Calling**：模型决定调用哪个预先注册的函数。
- **schema**：参数的名称、类型、范围和必填规则。
- **call_id**：把一次工具结果对应回原调用的标识。
- **只读/写工具**：是否会改变数据库状态。
- **显式确认**：用户点击后才允许写入。
- **最大轮数**：Agent 的安全停止条件。
- **规则降级**：不依赖模型也能生成结果。

## 7. 为什么这样设计

项目没有引入大型 Agent 框架，而是使用模型 SDK 的原生工具调用。代码更少、调用轨迹更容易解释，也方便用 fake client 测试。写工具和读工具共用注册表，但写入权限由页面确认按钮单独开启，模型本身拿不到直接写入权。

## 8. 常见错误与排查

1. **一直显示规则模式**：检查 API key 和模型名是否同时配置。
2. **工具参数无效**：查看工具摘要，检查城市、物品 ID 和动作类型。
3. **达到调用上限**：缩小请求，不要一次要求过多无关任务。
4. **确认后没有新增记录**：同一物品、动作和日期不会重复写入。
5. **留空周期仍提醒**：确认数据库迁移已运行并刷新页面。
6. **搜索无结果**：清空搜索框，或用名称、分类、备注中的一部分。

## 9. 面试可能追问

**问：Tool Calling 和把数据拼进 prompt 有什么区别？** 答：模型先根据意图选择必要工具，应用负责校验与执行，数据来源和轨迹更清楚。

**问：如何防止模型误写数据库？** 答：模型调用写工具只返回待确认动作；页面确认后才以 `allow_write=True` 执行。

**问：如何避免无限循环？** 答：代码限制最多 5 轮，超限后安全停止。

**问：外部服务失败怎么办？** 答：读取本地物品并用确定性规则生成计划。

## 10. 必须掌握的最少知识

模型只提出要调用的工具；Python 才真正执行。schema 拦截不合法参数；写入必须用户确认；循环有上限；没有模型时规则模式仍能工作。

## 11. 可自测小题

1. 哪两个工具是只读的？
2. 模型调用完成任务工具后，数据库会立刻变化吗？
3. `call_id` 有什么作用？
4. 为什么需要最大轮数？
5. 周期留空后会怎样？

<details><summary>参考答案</summary>

1. 天气和物品查询。2. 不会，要再确认。3. 对应调用与结果。4. 防止无限调用。5. 不参与到期提醒，但已有记录的天数仍显示。

</details>

## 12. 动手小练习

1. 新增一件物品，把清洗周期留空、晾晒周期填 7，观察两个状态。
2. 切换清洗/晾晒和升序/降序，观察无记录物品始终排在末尾。
3. 在智能助手输入“安排今天的洗晒”，展开工具摘要。

## 13. 本节点术语表

| 术语 | 简单解释 |
|---|---|
| function tool | 允许模型请求调用的本地函数 |
| registry | 集中保存和执行允许工具的入口 |
| argument validation | 在执行前检查参数 |
| pending action | 等待用户确认的操作 |
| idempotency | 重复提交不产生重复记录 |
| trace | 展示调用了哪些工具的摘要 |

## 14. 下一节点连接

下一节点会完善任务执行历史、重复提交提示和端到端验证，使手动勾选与 Agent 确认共享同一套可靠写入闭环。